In [2]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, TensorDataset, random_split
from transformers import T5Tokenizer, T5ForConditionalGeneration
import json
import random

c:\Users\Ashish\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
with open('../training-data/Sample-Resume.json', 'r', encoding='utf-8') as file:
  sample_resumes = json.load(file)

print(json.dumps(sample_resumes[0], indent=4))

{
    "fullName": "Aryan Naik",
    "jobTitle": "Software Engineer",
    "contact": "+91 9876543210",
    "location": "Mumbai, India",
    "email": "aryan.naik@example.com",
    "summary": "I'm a Software Engineer specializing in full-stack development and AI. With experience in designing scalable applications and optimizing backend processes, I create high-performing solutions that enhance user experience and business growth.",
    "experience": [
        {
            "company": "TechCorp",
            "role": "Software Engineer",
            "place": "Mumbai, India",
            "duration": "01/06/2020 - Present",
            "description": [
                "Developed scalable web applications and led backend optimizations.",
                "Implemented microservices architecture, improving system efficiency.",
                "Mentored junior developers, conducting regular code reviews."
            ]
        },
        {
            "company": "StartupX",
            "role": "In

In [4]:
num_epochs = 3
batch_size = 4

In [5]:
def create_training_data(resumes):
  training_pairs = []

  for resume in resumes:
    input_data = {
      'fullName': resume.get('fullName', ''),
      'jobTitle': resume.get('jobTitle', ''),
      'contact': resume.get('contact', ''),
      'location': resume.get('location', ''),
      'email': resume.get('email', ''),
      'experience': [
        {
          'company': exp.get('company', ''),
          'role': exp.get('role', ''),
          'place': exp.get('place', ''),
          'duration': exp.get('duration', ''),
        } for exp in resume.get('experience', [])
      ],
      'education': [
        {
          'institution': edu.get('institution', ''),
          'degree': edu.get('degree', ''),
          'year_end': edu.get('year_end', ''),
        }
        for edu in resume.get('education', [])
      ],
      'skills': sum([cat['skills'] for cat in resume.get('skills', [])], []),
    }

    output_data = resume

    training_pairs.append((input_data, output_data))
  
  return training_pairs

training_data = create_training_data(sample_resumes)
training_data = training_data[:10]

print("Input (Minimal Fields):", json.dumps(training_data[0][0], indent=4))
print("Output (Full Resume):", json.dumps(training_data[0][1], indent=4))

Input (Minimal Fields): {
    "fullName": "Aryan Naik",
    "jobTitle": "Software Engineer",
    "contact": "+91 9876543210",
    "location": "Mumbai, India",
    "email": "aryan.naik@example.com",
    "experience": [
        {
            "company": "TechCorp",
            "role": "Software Engineer",
            "place": "Mumbai, India",
            "duration": "01/06/2020 - Present"
        },
        {
            "company": "StartupX",
            "role": "Intern",
            "place": "Bangalore, India",
            "duration": "01/01/2020 - 31/05/2020"
        }
    ],
    "education": [
        {
            "institution": "Ramrao Adik Institute of Technology",
            "degree": "BTech in AI & Data Science",
            "year_end": "2026"
        }
    ],
    "skills": [
        "Software Development",
        "Machine Learning",
        "Data Science",
        "Python",
        "TensorFlow",
        "React.js",
        "Node.js",
        "Communication",
        "Problem-S

In [6]:
tokenizer = T5Tokenizer.from_pretrained('t5-small')

def tokenize_data(training_pairs):
  tokenized_data = []

  for input_data, output_data in training_pairs:
    input_text = f"Name: {input_data['fullName']}, Job: {input_data['jobTitle']}, Contact: {input_data['contact']}, Location: {input_data['location']}, Email: {input_data['email']}. Experience: " + "; ".join([f"{exp['role']} at {exp['company']} ({exp['duration']})" for exp in input_data['experience']]) + ". Education: " + "; ".join([f"{edu['degree']} from {edu['institution']} ({edu['year_end']})" for edu in input_data['education']]) + ". Skills: " + ", ".join(input_data['skills'])

    output_text = json.dumps(output_data)

    tokenized_input = tokenizer(input_text, padding='max_length', truncation=True, max_length=512, return_tensors='pt')
    tokenized_output = tokenizer(output_text, padding='max_length', truncation=True, max_length=512, return_tensors='pt')

    tokenized_data.append((tokenized_input, tokenized_output))

  return tokenized_data


tokenized_training_data = tokenize_data(training_data)

print('Tokenized Input IDs:', tokenized_training_data[0][0]['input_ids'].shape)
print('Tokenized Output IDs:', tokenized_training_data[0][1]['input_ids'].shape)

c:\Users\Ashish\AppData\Local\Programs\Python\Python312\Lib\site-packages\huggingface_hub\file_download.py:144: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Ashish\.cache\huggingface\hub\models--t5-small. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
You are using the default legacy behaviour of the <class 'transformers.models.t5.tokenization_t5.T5Token

Tokenized Input IDs: torch.Size([1, 512])
Tokenized Output IDs: torch.Size([1, 512])


In [8]:
class ResumeDataset(Dataset):
  def __init__(self, tokenized_data):
    self.input_ids = [pair[0]['input_ids'].squeeze(0) for pair in tokenized_data]
    self.attention_mask = [pair[0]['attention_mask'].squeeze(0) for pair in tokenized_data]
    self.labels = [pair[1]['input_ids'].squeeze(0) for pair in tokenized_data]

  def __len__(self):
    return len(self.input_ids)
  
  def __getitem__(self, idx):
    return {
      'input_ids': self.input_ids[idx],
      'attention_mask': self.attention_mask[idx],
      'labels': self.labels[idx],
    }
  
resume_dataset = ResumeDataset(tokenized_training_data)
train_dataloader = DataLoader(resume_dataset, batch_size=batch_size, shuffle=True)

batch = next(iter(train_dataloader))
print("Batch Input IDs Shape:", batch['input_ids'].shape)
print("Batch Labels Shape:", batch['labels'].shape)
    

Batch Input IDs Shape: torch.Size([4, 512])
Batch Labels Shape: torch.Size([4, 512])


In [10]:
model = T5ForConditionalGeneration.from_pretrained('t5-small')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)

print('Model loaded and moved to:', device)

Model loaded and moved to: cpu


In [12]:
# Step 4: Prepare training data
#     Step 4.1: Convert text into tokenized input-output pairs
#     Step 4.2: Create training and validation data splits
#     Step 4.3: Define DataLoader for batch processing

input_ids = torch.cat([pair[0]['input_ids'] for pair in tokenized_training_data])
attention_masks = torch.cat([pair[0]['attention_mask'] for pair in tokenized_training_data])
labels = torch.cat([pair[1]['input_ids'] for pair in tokenized_training_data])

dataset = TensorDataset(input_ids, attention_masks, labels)

train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size
train_dataset, val_dataset = random_split(dataset, [train_size, val_size])

train_dataloader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

print(f"Total samples: {len(dataset)} | Training: {len(train_dataset)} | Validation: {len(val_dataset)}")

Total samples: 10 | Training: 8 | Validation: 2


In [ ]:
# Better to run on google collab

loss_function = nn.CrossEntropyLoss(ignore_index=tokenizer.pad_token_id)

for epoch in range(num_epochs):
  model.train()
  total_loss = 0

  for batch in train_dataloader:
    input_ids, attention_masks, labels = batch

    outputs = model(input_ids=input_ids, attention_mask=attention_masks, labels=labels)
    loss = loss_function(outputs.logits.view(-1, outputs.logits.size(-1)), labels.view(-1))

    total_loss += loss.item()

  avg_loss = total_loss / len(train_dataloader)
  print(f"Epoch: {epoch + 1}/{num_epochs} - Loss: {avg_loss:.4f}")

print("Training (without optimization) completed.")

Passing a tuple of `past_key_values` is deprecated and will be removed in Transformers v4.48.0. You should pass an instance of `EncoderDecoderCache` instead, e.g. `past_key_values=EncoderDecoderCache.from_legacy_cache(past_key_values)`.


Epoch: 1/3 - Loss: 12.1171


In [40]:
print('GPU Available:', torch.cuda.is_available())

GPU Available: False
